# MLOps Pipeline Walkthrough - End-to-End Demo

**Project:** MLOps-End-to-End-Pipeline  
**Task:** Sentiment Analysis (TF-IDF + ML classifiers)  
**Stack:** scikit-learn, MLflow, FastAPI, pytest, Docker  
**Alignment:** BP-ITCS MLOps Engineering  

---

This notebook walks through every stage of the production ML pipeline, demonstrating
reproducible data versioning, feature engineering, model training with experiment
tracking, automated quality gates, model registry management, and API serving.

## Overview of the MLOps Lifecycle

The pipeline follows a six-stage lifecycle that mirrors production MLOps best practices:

```
┌──────────────┐    ┌───────────────┐    ┌──────────────┐
│  1. Data      │───▶│  2. Feature   │───▶│  3. Train    │
│  Pipeline     │    │  Store        │    │  (+ MLflow)  │
└──────────────┘    └───────────────┘    └──────┬───────┘
                                                │
                                                ▼
┌──────────────┐    ┌───────────────┐    ┌──────────────┐
│  6. Serve    │◀───│  5. Model     │◀───│  4. Evaluate │
│  (FastAPI)   │    │  Registry     │    │  (+ Gates)   │
└──────────────┘    └───────────────┘    └──────────────┘
```

| Stage | Module | Responsibility |
|-------|--------|----------------|
| **Data Pipeline** | `src.data_pipeline` | Load, clean, validate, and version raw data |
| **Feature Store** | `src.feature_store` | TF-IDF vectorization + numeric scaling with caching |
| **Training** | `src.train` | Model training, cross-validation, MLflow logging |
| **Evaluation** | `src.evaluate` | Metrics computation, performance gates (accuracy, F1, latency) |
| **Model Registry** | `src.model_registry` | Version-controlled model storage with stage transitions |
| **Serving** | `src.serve` | FastAPI REST endpoint for real-time inference |

## Imports and Setup

We add the project root to `sys.path` so that `src.*` modules can be imported
directly from this notebook.

In [ ]:
import sys
import os

# Add project root so that `src.*` imports resolve correctly
sys.path.insert(0, '..')

# Standard libraries
import json
import warnings
warnings.filterwarnings('ignore')

# Data & ML
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.grid'] = True

print(f"Python: {sys.version}")
print(f"NumPy:  {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Working dir: {os.getcwd()}")

---

## Step 1 - Data Pipeline

The data pipeline (`src.data_pipeline`) handles:
- **Loading** raw review data (or generating synthetic demo data)
- **Text preprocessing** — lowercasing, URL/HTML removal, whitespace normalization
- **Data versioning** — SHA-256 content hash for reproducibility
- **Validation** — null checks, duplicate detection, class-balance analysis

In [ ]:
from src.data_pipeline import (
    load_and_preprocess,
    compute_data_hash,
    DataValidator,
    preprocess_text,
)

# Load config (same config used by CLI scripts)
import yaml

with open('../configs/train_config.yaml') as f:
    config = yaml.safe_load(f)

print("=== Configuration ===")
print(json.dumps(config, indent=2))

In [ ]:
# Run the data pipeline: load, clean, version, validate
df, data_hash = load_and_preprocess(config)

print(f"\n{'='*50}")
print(f"Data hash (SHA-256 prefix): {data_hash}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"{'='*50}")

df.head()

In [ ]:
# Run full data validation
validator = DataValidator(config)
label_col = config['data']['label_column']

validation_report = validator.run_all_checks(df, label_col)

print("=== Data Validation Report ===")
print(f"Null check passed:       {validation_report['null_check']['passed']}")
print(f"Duplicate check passed:  {validation_report['duplicate_check']['passed']}")
print(f"Duplicate %:             {validation_report['duplicate_check']['duplicate_percentage']:.2%}")

if validation_report['class_balance']:
    cb = validation_report['class_balance']
    print(f"Class balance passed:    {cb['passed']}")
    print(f"Class distribution:      {cb['class_distribution']}")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df[label_col].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Sentiment Class Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Review length distribution
df['review_length'].hist(bins=30, ax=axes[1], color='#3498db', alpha=0.7, edgecolor='black')
axes[1].set_title('Review Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

---

## Step 2 - Feature Store

The feature store (`src.feature_store`) computes and caches:
- **TF-IDF features** — uni/bigrams with sublinear TF, configurable `max_features`
- **Numeric features** — `review_length`, `word_count` scaled via `StandardScaler`
- **Hash-based caching** — avoids recomputation when data hasn't changed

In [ ]:
from src.feature_store import FeatureStore

# Prepare train/test splits
y = df[label_col].values
test_size = config['data'].get('test_size', 0.2)
random_state = config['data'].get('random_state', 42)

df_train, df_test, y_train, y_test = train_test_split(
    df, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y,
)

print(f"Train set: {df_train.shape[0]} samples")
print(f"Test set:  {df_test.shape[0]} samples")

In [ ]:
# Compute features using the FeatureStore
feature_store = FeatureStore(config)
X_train, X_test = feature_store.get_features(df_train, df_test, use_cache=False)

print(f"\n{'='*50}")
print(f"Training feature matrix shape: {X_train.shape}")
print(f"Test feature matrix shape:     {X_test.shape}")
print(f"Feature matrix type:           {type(X_train).__name__}")
print(f"{'='*50}")

# Feature store metadata
metadata = feature_store.metadata
print(f"\n=== Feature Store Metadata ===")
for key, value in metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Save feature transformers for serving (TF-IDF vectorizer + scaler)
feature_store.save_transformers(path='../models/feature_transformers.pkl')

# Inspect top TF-IDF features by weight
if feature_store.tfidf is not None:
    feature_names = feature_store.tfidf.get_feature_names_out()
    # Average TF-IDF weight across training documents
    avg_tfidf = np.array(X_train[:, :len(feature_names)].mean(axis=0)).flatten()
    top_indices = avg_tfidf.argsort()[-15:][::-1]

    print("\nTop 15 TF-IDF features by average weight:")
    for i, idx in enumerate(top_indices, 1):
        print(f"  {i:2d}. {feature_names[idx]:<25s} weight={avg_tfidf[idx]:.4f}")

---

## Step 3 - Training

The training module (`src.train`) supports:
- **Model factory** — Logistic Regression, Random Forest, SVM
- **Stratified k-fold cross-validation** for robust metric estimation
- **MLflow experiment tracking** (optional — graceful fallback if unavailable)
- **Metric logging** — accuracy, F1, precision, recall, training time

In [ ]:
from src.train import build_model, cross_validate_model, train_model, save_model

# Train the model using the full training pipeline
model, metrics = train_model(
    X_train, y_train,
    X_test, y_test,
    config,
    experiment_name='notebook-walkthrough',
)

print(f"\n{'='*50}")
print("=== Training Results ===")
print(f"{'='*50}")
print(f"Model type:       {config['model']['type']}")
print(f"Test Accuracy:    {metrics['accuracy']:.4f}")
print(f"Test F1 (weighted): {metrics['f1_weighted']:.4f}")
print(f"Precision:        {metrics['precision_weighted']:.4f}")
print(f"Recall:           {metrics['recall_weighted']:.4f}")
print(f"Training time:    {metrics['train_time_seconds']:.2f}s")
print(f"\nCross-Validation ({config['training']['cv_folds']}-fold):")
print(f"  Mean F1: {metrics['cv_mean']:.4f} +/- {metrics['cv_std']:.4f}")
print(f"  Fold scores: {[round(s, 4) for s in metrics['cv_scores']]}")

In [ ]:
# Save model artifacts to disk
save_model(model, metrics, path='../models/latest')

# Visualize cross-validation scores
fig, ax = plt.subplots(figsize=(8, 4))
folds = range(1, len(metrics['cv_scores']) + 1)
bars = ax.bar(folds, metrics['cv_scores'], color='#3498db', alpha=0.8, edgecolor='black')
ax.axhline(y=metrics['cv_mean'], color='#e74c3c', linestyle='--', linewidth=2,
           label=f"Mean = {metrics['cv_mean']:.4f}")
ax.fill_between(
    [0.5, len(folds) + 0.5],
    metrics['cv_mean'] - metrics['cv_std'],
    metrics['cv_mean'] + metrics['cv_std'],
    alpha=0.15, color='#e74c3c', label=f'+/- 1 std ({metrics["cv_std"]:.4f})',
)
ax.set_xlabel('Fold')
ax.set_ylabel('F1 (weighted)')
ax.set_title('Stratified K-Fold Cross-Validation Results')
ax.set_xticks(list(folds))
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 4 - Evaluation

The evaluation module (`src.evaluate`) provides:
- **ModelEvaluator** — comprehensive metrics: accuracy, F1, confusion matrix, ROC-AUC, PR curve, latency
- **PerformanceGate** — automated go/no-go decision based on configurable thresholds:
  - `min_accuracy` (default 0.85)
  - `min_f1` (default 0.83)
  - `max_latency_ms` (default 100ms at p95)

In [ ]:
from src.evaluate import ModelEvaluator, PerformanceGate, generate_report

# Evaluate model on the held-out test set
evaluator = ModelEvaluator(model, config)
eval_metrics = evaluator.compute_metrics(X_test, y_test)

print("=== Evaluation Metrics ===")
print(f"Accuracy:    {eval_metrics['accuracy']:.4f}")
print(f"F1 weighted: {eval_metrics['f1_weighted']:.4f}")
if 'roc_auc' in eval_metrics:
    print(f"ROC-AUC:     {eval_metrics['roc_auc']:.4f}")

# Confusion matrix
print(f"\nConfusion Matrix:")
cm = np.array(eval_metrics['confusion_matrix'])
print(cm)

In [ ]:
# Measure inference latency
latency = evaluator.measure_latency(X_test, n_runs=100)

print("=== Inference Latency ===")
print(f"Mean:  {latency['mean_latency_ms']:.3f} ms")
print(f"P95:   {latency['p95_latency_ms']:.3f} ms")
print(f"P99:   {latency['p99_latency_ms']:.3f} ms")

In [ ]:
# Run performance gates
gate = PerformanceGate(config)
gate_result = gate.evaluate(eval_metrics, latency)

print("=== Performance Gate Results ===")
print(f"Overall PASSED: {gate_result['overall_passed']}")
print()
for check_name, check_info in gate_result['checks'].items():
    status = 'PASS' if check_info['passed'] else 'FAIL'
    print(f"  [{status}] {check_name:<15s}  value={check_info['value']:.4f}  "
          f"threshold={check_info['threshold']}")

# Save evaluation report
report_path = generate_report(eval_metrics, latency, gate_result, output_dir='../results')
print(f"\nReport saved to: {report_path}")

In [ ]:
# Plot ROC curve and confusion matrix side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
if 'roc_curve' in eval_metrics:
    fpr = eval_metrics['roc_curve']['fpr']
    tpr = eval_metrics['roc_curve']['tpr']
    axes[0].plot(fpr, tpr, color='#3498db', linewidth=2,
                 label=f"AUC = {eval_metrics['roc_auc']:.4f}")
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random baseline')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve')
    axes[0].legend(loc='lower right')
else:
    axes[0].text(0.5, 0.5, 'ROC curve not available\n(multiclass or no predict_proba)',
                 ha='center', va='center', transform=axes[0].transAxes)

# Confusion Matrix
im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
axes[1].set_title('Confusion Matrix')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
# Annotate cells
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black',
                     fontsize=14, fontweight='bold')
classes = sorted(np.unique(y_test))
axes[1].set_xticks(range(len(classes)))
axes[1].set_yticks(range(len(classes)))
axes[1].set_xticklabels(classes)
axes[1].set_yticklabels(classes)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

---

## Step 5 - Model Registry

The model registry (`src.model_registry`) provides:
- **LocalModelRegistry** — file-system-based registry with JSON index
- **Version management** — auto-incrementing version tags (`v1`, `v2`, ...)
- **Stage transitions** — `none` → `staging` → `production` → `archived`
- **Artifact lineage** — SHA-256 hash of model binaries for traceability

In [ ]:
from src.model_registry import LocalModelRegistry

# Initialize local registry
registry = LocalModelRegistry(registry_dir='../models/registry')

# Determine initial stage based on gate results
initial_stage = 'staging' if gate_result['overall_passed'] else 'none'

# Register the model
model_name = 'sentiment-classifier'
version_tag = registry.register_model(
    model_name=model_name,
    model_path='../models/latest',
    metrics=metrics,
    stage=initial_stage,
    description=f'Notebook walkthrough demo | data_hash={data_hash}',
)

print(f"\nRegistered: {model_name} {version_tag} (stage={initial_stage})")

In [ ]:
# If gates passed, promote to production
if gate_result['overall_passed']:
    registry.transition_stage(model_name, version_tag, 'production')
    print(f"Promoted {model_name} {version_tag} to PRODUCTION")
else:
    print(f"Model did NOT pass gates - remains at stage '{initial_stage}'")

# List all registered versions
print(f"\n{'='*60}")
print(f"{'Version':<10} {'Stage':<14} {'Accuracy':<12} {'F1':<12} {'Registered At'}")
print(f"{'-'*60}")
for v in registry.list_versions(model_name):
    acc = v['metrics'].get('accuracy', 'N/A')
    f1 = v['metrics'].get('f1_weighted', 'N/A')
    acc_str = f"{acc:.4f}" if isinstance(acc, float) else str(acc)
    f1_str = f"{f1:.4f}" if isinstance(f1, float) else str(f1)
    print(f"{v['version']:<10} {v['stage']:<14} {acc_str:<12} {f1_str:<12} {v['registered_at']}")
print(f"{'='*60}")

---

## Step 6 - Serving Demo

The serving module (`src.serve`) exposes a **FastAPI** application with:
- `GET /health` — health check for Docker `HEALTHCHECK` and load balancers
- `POST /predict` — single-text sentiment prediction
- `POST /predict/batch` — batch prediction (up to 100 texts)
- `GET /metrics` — runtime prediction statistics

### Request/Response Schema

```json
// POST /predict
{
  "text": "This product is amazing, I love it!",
  "review_length": 35,
  "word_count": 7
}

// Response
{
  "prediction": "positive",
  "confidence": 0.9412,
  "latency_ms": 1.234,
  "model_version": "acc=0.9350"
}
```

In [ ]:
from src.serve import PredictRequest, BatchPredictRequest

# Demonstrate how to construct request payloads
# (In production, these are sent as HTTP POST to the FastAPI server)

# --- Single prediction request ---
single_request = PredictRequest(
    text="This product is excellent and works great, highly recommended!",
    review_length=60,
    word_count=9,
)

print("=== Single Prediction Request ===")
print(json.dumps(single_request.model_dump(), indent=2))

# --- Batch prediction request ---
batch_request = BatchPredictRequest(
    texts=[
        "Absolutely love this product, best purchase ever!",
        "Terrible quality, broke after one day. Waste of money.",
        "It is okay, nothing special but does the job.",
    ]
)

print("\n=== Batch Prediction Request ===")
print(json.dumps(batch_request.model_dump(), indent=2))

In [ ]:
# Simulate predictions using the AppState class directly
# (This mirrors what the FastAPI endpoint does internally)
from src.serve import AppState

app_state = AppState()
app_state.load('../models/latest')

# Run predictions on sample texts
test_texts = [
    "This product is excellent and works great, highly recommended!",
    "Terrible quality, broke after one day. Waste of money.",
    "It is okay, nothing special but does the job.",
    "Very happy with the quality and fast delivery",
    "Horrible experience would not recommend to anyone",
]

print(f"{'Text':<60s} {'Prediction':<12s} {'Confidence':<12s} {'Latency'}")
print('-' * 100)
for text in test_texts:
    result = app_state.predict_single(text)
    truncated = text[:57] + '...' if len(text) > 57 else text
    print(f"{truncated:<60s} {result['prediction']:<12s} "
          f"{result['confidence']:<12.4f} {result['latency_ms']:.3f} ms")

### Deployment with Docker

The project includes a `Dockerfile` and `docker-compose.yml` for containerized deployment:

```bash
# Build and run the inference server
docker-compose up --build

# Test the endpoint
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "Great product!", "review_length": 14, "word_count": 2}'

# Health check
curl http://localhost:8000/health

# Interactive API docs
open http://localhost:8000/docs
```

The server loads the production model on startup via the `MODEL_PATH` environment variable
and exposes endpoints on port 8000 with automatic OpenAPI documentation.

---

## Pipeline Architecture

```
                          MLOps-End-to-End-Pipeline
    ┌─────────────────────────────────────────────────────────────────┐
    │                                                                 │
    │   configs/                                                      │
    │   └── train_config.yaml  ─────────────────────────┐            │
    │                                                    │            │
    │   src/                                             ▼            │
    │   ├── data_pipeline.py   ──── Load / Clean / Hash / Validate   │
    │   │        │                                                    │
    │   │        ▼                                                    │
    │   ├── feature_store.py   ──── TF-IDF + Numeric + Cache         │
    │   │        │                                                    │
    │   │        ▼                                                    │
    │   ├── train.py           ──── Model Factory + CV + MLflow       │
    │   │        │                                                    │
    │   │        ▼                                                    │
    │   ├── evaluate.py        ──── Metrics + Performance Gates       │
    │   │        │                                                    │
    │   │        ▼                                                    │
    │   ├── model_registry.py  ──── Version + Stage Management        │
    │   │        │                                                    │
    │   │        ▼                                                    │
    │   └── serve.py           ──── FastAPI + Docker Deployment       │
    │                                                                 │
    │   tests/                                                        │
    │   ├── test_data_pipeline.py                                     │
    │   ├── test_model.py                                             │
    │   └── test_api.py                                               │
    │                                                                 │
    │   Dockerfile + docker-compose.yml                               │
    └─────────────────────────────────────────────────────────────────┘

    Key data artifacts:
    ┌──────────────┐   ┌───────────────────┐   ┌────────────────┐
    │ data/        │   │ models/            │   │ results/       │
    │  manifest_*  │   │  latest/           │   │  evaluation_   │
    │  processed   │   │   model.joblib     │   │  report.json   │
    │  feature_    │   │   metrics.json     │   └────────────────┘
    │  cache/      │   │  feature_trans.pkl │
    └──────────────┘   │  registry/         │
                       │   registry.json    │
                       └───────────────────┘
```

---

## Summary

This notebook demonstrated the full MLOps lifecycle for a sentiment analysis pipeline:

| Practice | Implementation |
|----------|---------------|
| **Data Versioning** | SHA-256 content hashing with JSON manifests for reproducibility |
| **Data Validation** | Automated null checks, duplicate detection, class-balance analysis, drift detection (PSI + KS-test) |
| **Feature Engineering** | TF-IDF (uni/bigram, sublinear TF) + scaled numeric features with hash-based caching |
| **Experiment Tracking** | MLflow integration for parameters, metrics, and model artifacts |
| **Cross-Validation** | Stratified k-fold CV for robust, unbiased metric estimation |
| **Performance Gates** | Automated go/no-go checks on accuracy, F1, and inference latency before promotion |
| **Model Registry** | Version-controlled storage with stage transitions (none → staging → production → archived) |
| **Serving** | FastAPI with health checks, single/batch prediction, prediction logging, and Docker deployment |
| **Testing** | pytest test suite covering data pipeline, model training, and API endpoints |
| **Reproducibility** | Deterministic seeds, config-driven pipeline, artifact lineage via content hashing |

### Next Steps

- **CI/CD**: Integrate `pytest` and performance gates into GitHub Actions
- **Monitoring**: Add Prometheus metrics and Grafana dashboards for production drift detection
- **A/B Testing**: Use the model registry to serve multiple model versions behind a feature flag
- **Retraining**: Schedule automated retraining when data drift PSI exceeds threshold